# Try_017: Pre-GPTQ LoRA 정렬 + GPTQ W4A16

**전략: 양자화 전 LoRA로 가중치를 평가 분포에 정렬 → GPTQ 오류 감소**

### 핵심 아이디어 (QAT-inspired)
GPTQ는 캘리브레이션 데이터 기준으로 Hessian을 계산해 양자화 오류를 최소화합니다.  
그러나 *모델 가중치 자체가 해당 분포와 잘 맞지 않으면* 양자화 후 성능이 떨어집니다.

**해결책**: GPTQ 전에 LoRA 파인튜닝으로 모델 가중치를 MANTA+KMMLU 분포에 정렬  
→ Hessian이 더 정확하게 추정되고 양자화 오류가 줄어듦

### Try_014 vs Try_017
| 항목 | Try_014 | Try_017 |
|------|---------|----------|
| 양자화 | GPTQ W4A16 | **LoRA 정렬 → GPTQ W4A16** |
| 캘리브레이션 | MANTA 128 + KMMLU 32 | MANTA 128 + KMMLU 32 (동일) |
| LoRA | 없음 | **r=16, 200 steps (정렬용)** |
| 예상 PerfNorm | Try_014보다 낮음 or 비슷 | **Try_014 이상 기대** |
| 예상 SpeedNorm | 기준 | 동일 (LoRA 병합 후 제거) |

### 파이프라인
```
FP16 모델 로드
  → LoRA 파인튜닝 (MANTA+KMMLU, r=16, 200 steps)
  → LoRA 병합 (merge_and_unload)
  → 생성 검증 (unique_ratio ≥ 0.3)
  → GPTQ W4A16 양자화
  → 생성 검증
  → tie_weights() + 저장
  → zip 생성
```

예상 소요 시간: ~3-4시간 (MPS) / ~1시간 (CUDA GPU)

In [1]:
import os
import gc
import itertools
import torch
import shutil
from pathlib import Path

from datasets import load_dataset, Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments

from peft import LoraConfig, TaskType
from trl import SFTTrainer, SFTConfig

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

import psutil
ram = psutil.virtual_memory()
print(f"전체 RAM: {ram.total / 1e9:.1f} GB  |  사용 가능: {ram.available / 1e9:.1f} GB")
print(f"[INFO] TRL 버전: {__import__('trl').__version__}")
print("[INFO] 임포트 완료")

전체 RAM: 25.8 GB  |  사용 가능: 9.3 GB
[INFO] TRL 버전: 0.14.0
[INFO] 임포트 완료


In [2]:
# =============================================
# 환경 감지 + 패키지 설치 (Colab/로컬 자동 분기)
# ─────────────────────────────────────────────
# [에러 수정] ImportError: download_url from transformers.utils
#   원인: Colab 기본 TRL이 구버전 → 신버전 transformers와 충돌
#   수정: llmcompressor + trl>=0.9.0 설치로 호환 버전 강제
# =============================================
import os, sys, subprocess

IS_COLAB = os.path.exists("/content")
print(f"[INFO] 실행 환경: {'Google Colab' if IS_COLAB else '로컬'}")

if IS_COLAB:
    print("[INFO] 필수 패키지 설치 중...")
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "llmcompressor==0.9.0.2",  # 양자화 라이브러리 (호환 transformers 자동 설치)
        "trl>=0.9.0",              # ← download_url 오류 수정 핵심
        "peft>=0.6.0",
        "accelerate",
    ])
    print("[INFO] 패키지 설치 완료")

    from google.colab import drive
    drive.mount("/content/drive")
    print("[INFO] Google Drive 마운트 완료")
else:
    print("[INFO] 로컬 환경 — 패키지 설치 스킵")

[INFO] 실행 환경: 로컬
[INFO] 로컬 환경 — 패키지 설치 스킵


In [3]:
# =============================================
# GPU 디바이스 자동 감지
# - 모델 로드 / LoRA 학습 / 생성 검증: MPS 또는 CUDA 사용
# - GPTQ oneshot(): CUDA만 지원 → CUDA 없으면 CPU 강제
# =============================================
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"[INFO] ✅ CUDA GPU 감지: {torch.cuda.get_device_name(0)}")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("[INFO] ✅ Apple MPS (GPU) 감지")
else:
    DEVICE = torch.device("cpu")
    print("[INFO] CPU 사용")

print(f"[INFO] DEVICE = {DEVICE}")
print("[INFO] ⚠️  GPTQ(oneshot)는 CUDA만 지원 → MPS 환경에서는 GPTQ 단계만 CPU 실행")

[INFO] ✅ Apple MPS (GPU) 감지
[INFO] DEVICE = mps
[INFO] ⚠️  GPTQ(oneshot)는 CUDA만 지원 → MPS 환경에서는 GPTQ 단계만 CPU 실행


In [4]:
# =============================================
# 하이퍼파라미터
# =============================================
# IS_COLAB은 첫 번째 셀에서 정의됨 (없을 경우 대비 fallback)
try:
    IS_COLAB
except NameError:
    IS_COLAB = os.path.exists("/content")

if IS_COLAB:
    MODEL_ID = "/content/drive/MyDrive/base_model"
    OUT_DIR  = "/content/drive/MyDrive/model"
else:
    MODEL_ID = "./base_model"
    OUT_DIR  = "./model"

# ─── 캘리브레이션 / LoRA 학습 데이터 ────────
N_MANTA = 128
N_KMMLU = 32
NUM_CALIBRATION_SAMPLES = N_MANTA + N_KMMLU  # 160
MAX_SEQUENCE_LENGTH = 512

KMMLU_SUBJECTS = [
    "Korean-History", "Law", "Computer-Science",
    "Management", "Health", "Education"
]

# ─── LoRA 설정 (정렬용, 가벼운 파인튜닝) ────
# 목적: GPTQ 전 가중치를 평가 분포에 정렬
# 과적합 방지: 짧은 steps + 낮은 rank
LORA_R        = 16       # rank (낮을수록 빠름, 높을수록 표현력)
LORA_ALPHA    = 32       # lora_alpha = 2 * rank 가 일반적
LORA_DROPOUT  = 0.05
LORA_STEPS    = 200      # 정렬 목적 → 짧게 (과적합 방지)
LORA_LR       = 1e-4
LORA_BATCH    = 1        # Mac 메모리 제약
LORA_GRAD_ACC = 8        # effective batch = 8

# LoRA 적용 대상: 모든 선형 레이어
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj"
]

# ─── GPTQ 설정 ──────────────────────────────
SCHEME  = "W4A16"
TARGETS = ["Linear"]
IGNORE  = ["embed_tokens", "lm_head"]  # tie_word_embeddings=True

print("[INFO] 설정 완료")
print(f"  환경: {'Colab' if IS_COLAB else '로컬'}")
print(f"  MODEL_ID: {MODEL_ID}")
print(f"  OUT_DIR:  {OUT_DIR}")
print(f"  LoRA: r={LORA_R}, alpha={LORA_ALPHA}, steps={LORA_STEPS}, lr={LORA_LR}")
print(f"  캘리브레이션: MANTA {N_MANTA} + KMMLU {N_KMMLU} = {NUM_CALIBRATION_SAMPLES}개")
print(f"  양자화: GPTQ W4A16")

[INFO] 설정 완료
  환경: 로컬
  MODEL_ID: ./base_model
  OUT_DIR:  ./model
  LoRA: r=16, alpha=32, steps=200, lr=0.0001
  캘리브레이션: MANTA 128 + KMMLU 32 = 160개
  양자화: GPTQ W4A16


In [5]:
print("[INFO] 모델 로드 중 (bfloat16, CPU)...")
print("  LoRA 학습 전 CPU 로드 → MPS/CUDA로 학습 중 이동")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
    trust_remote_code=True,
)

print(f"[INFO] 로드 완료 | 레이어: {model.config.num_hidden_layers}")
print(f"  tie_word_embeddings: {model.config.tie_word_embeddings}")

[INFO] 모델 로드 중 (bfloat16, CPU)...
  LoRA 학습 전 CPU 로드 → MPS/CUDA로 학습 중 이동
[INFO] 로드 완료 | 레이어: 30
  tie_word_embeddings: True


In [6]:
# =============================================
# 캘리브레이션 / LoRA 학습 데이터 로드
# LoRA 학습과 GPTQ 캘리브레이션에 동일한 데이터 사용
# → 가중치-캘리브레이션 분포 정렬 극대화
# =============================================
print("[INFO] 데이터 로드 중...")
all_samples = []

# ── 1. MANTA 128개 ────────────────────────────
print(f"  [1/2] MANTA {N_MANTA}개 로드 중...")
manta_ds = load_dataset("LGAI-EXAONE/MANTA-1M", split=f"train[:{N_MANTA}]")
for ex in manta_ds:
    text = tokenizer.apply_chat_template(
        ex["conversations"], add_generation_prompt=True, tokenize=False
    )
    all_samples.append({"text": text})
print(f"      → {len(all_samples)}개")

# ── 2. KMMLU 32개 (과목별 순환) ───────────────
print(f"  [2/2] KMMLU {N_KMMLU}개 로드 중...")
per_subject = max(1, N_KMMLU // len(KMMLU_SUBJECTS))
kmmlu_total = 0

for subj in KMMLU_SUBJECTS:
    if kmmlu_total >= N_KMMLU:
        break
    n_this = min(per_subject, N_KMMLU - kmmlu_total)
    try:
        kds = load_dataset("HAERAE-HUB/KMMLU", subj, split=f"test[:{n_this}]")
        for item in kds:
            q = item.get("question", item.get("Question", ""))
            choices = "".join(
                f"\n{k}. {item[k]}" for k in ["A","B","C","D"] if item.get(k)
            )
            messages = [{"role": "user", "content": q + choices}]
            text = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False
            )
            all_samples.append({"text": text})
            kmmlu_total += 1
        print(f"      {subj}: {n_this}개")
    except Exception as e:
        print(f"      {subj}: 실패 ({e})")

if kmmlu_total < N_KMMLU:
    deficit = N_KMMLU - kmmlu_total
    extra_ds = load_dataset("LGAI-EXAONE/MANTA-1M", split=f"train[{N_MANTA}:{N_MANTA+deficit}]")
    for ex in extra_ds:
        text = tokenizer.apply_chat_template(
            ex["conversations"], add_generation_prompt=True, tokenize=False
        )
        all_samples.append({"text": text})
    print(f"      ⚠ KMMLU {deficit}개 부족 → MANTA 보충")

ds = Dataset.from_list(all_samples[:NUM_CALIBRATION_SAMPLES])
print(f"\n[INFO] 데이터 완료: {len(ds)}개 (LoRA 학습 + GPTQ 캘리브레이션 공용)")

[INFO] 데이터 로드 중...
  [1/2] MANTA 128개 로드 중...
      → 128개
  [2/2] KMMLU 32개 로드 중...


korean-history-train.csv: 0.00B [00:00, ?B/s]

korean-history-dev.csv: 0.00B [00:00, ?B/s]

korean-history-test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1 [00:00<?, ? examples/s]

...

Education-test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/2 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/5 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/100 [00:00<?, ? examples/s]

      Education: 5개
      ⚠ KMMLU 2개 부족 → MANTA 보충

[INFO] 데이터 완료: 160개 (LoRA 학습 + GPTQ 캘리브레이션 공용)


In [9]:
# =============================================
# LoRA 파인튜닝 — 가중치 분포 정렬
# TRL 0.28 API: peft_config → SFTTrainer 직접 전달
#               max_length, dataset_text_field → SFTConfig 소속
#               tokenizer → processing_class
# =============================================
print("[INFO] LoRA 설정 중 (TRL 0.28 API)...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=LORA_TARGET_MODULES,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ── 학습 디바이스 ──────────────────────────
model.to(DEVICE)
print(f"[INFO] LoRA 학습 디바이스: {DEVICE}")

# MPS: bf16 미지원 → fp32 폴백
use_bf16 = (str(DEVICE) == "cuda")

# ── SFTConfig (TRL 0.9+): 학습 설정 + SFT 전용 설정 통합 ──
sft_config = SFTConfig(
    output_dir="./lora_tmp",
    max_steps=LORA_STEPS,
    per_device_train_batch_size=LORA_BATCH,
    gradient_accumulation_steps=LORA_GRAD_ACC,
    learning_rate=LORA_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.05,
    bf16=use_bf16,
    fp16=False,
    logging_steps=20,
    save_steps=999999,
    report_to="none",
    dataloader_pin_memory=False,
    # ── SFT 전용 (TrainingArguments가 아닌 SFTConfig 소속) ──
    max_seq_length=MAX_SEQUENCE_LENGTH,   # max_seq_length → max_length
    dataset_text_field="text",
    packing=False,
)

# ── SFTTrainer: peft_config 직접 전달 (get_peft_model 불필요) ──
trainer = SFTTrainer(
    model=model,                      # base FP16 모델 그대로
    args=sft_config,
    train_dataset=ds,
    processing_class=tokenizer,       # tokenizer → processing_class
    peft_config=lora_config,          # LoRA 어댑터 자동 적용
)

print(f"[INFO] LoRA 학습 시작 ({LORA_STEPS} steps, lr={LORA_LR})...")
trainer.train()

# 학습 완료 후 trained model 참조 업데이트
model = trainer.model
print("[INFO] LoRA 학습 완료")

[INFO] LoRA 설정 중 (TRL 0.28 API)...
[INFO] LoRA 학습 디바이스: mps


Map:   0%|          | 0/160 [00:00<?, ? examples/s]

[INFO] LoRA 학습 시작 (200 steps, lr=0.0001)...


Step,Training Loss
20,1.565100
40,1.094200
60,0.935800
80,0.798400
100,0.659100
120,0.533300
140,0.427300
160,0.365100
180,0.326800
200,0.315400


[INFO] LoRA 학습 완료


In [10]:
# =============================================
# LoRA 병합 → FP16 단일 모델로 복원
# merge_and_unload(): LoRA 어댑터를 base 가중치에 흡수
# 이후에는 일반 모델처럼 동작 → GPTQ 양자화 적용 가능
# =============================================
print("[INFO] LoRA 병합 중 (merge_and_unload)...")

model = model.merge_and_unload()

gc.collect()
if str(DEVICE) == "mps":
    torch.mps.empty_cache()
elif str(DEVICE) == "cuda":
    torch.cuda.empty_cache()

print(f"[INFO] LoRA 병합 완료")
print(f"  레이어 수: {model.config.num_hidden_layers}")

# layer_types 확인
if hasattr(model.config, 'layer_types'):
    print(f"  layer_types: {len(model.config.layer_types)}개 (num_hidden_layers와 일치 필수)")

[INFO] LoRA 병합 중 (merge_and_unload)...
[INFO] LoRA 병합 완료
  레이어 수: 30
  layer_types: 30개 (num_hidden_layers와 일치 필수)


In [11]:
# =============================================
# 생성 검증 1 — LoRA 병합 후 (GPTQ 전)
# unique_ratio < 0.3 → 반복 루프 → LoRA steps 재조정 필요
# =============================================
print("[INFO] 생성 검증 (LoRA 병합 후)...")

test_prompts = [
    [{"role": "user", "content": "한국의 수도는 어디인가요?"}],
    [{"role": "user", "content": "다음 중 맞는 것은? A. 지구는 평평하다 B. 지구는 둥글다 C. 지구는 삼각형 D. 지구는 네모"}],
    [{"role": "user", "content": "Python에서 리스트를 정렬하는 함수는?"}],
]

model.eval()
model.to(DEVICE)

all_pass = True
for i, msgs in enumerate(test_prompts):
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=40, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    unique_ratio = len(set(gen_ids.tolist())) / max(len(gen_ids), 1)
    status = "✅" if unique_ratio >= 0.3 else "❌ 반복루프"
    if unique_ratio < 0.3:
        all_pass = False
    print(f"  [{i+1}] unique_ratio={unique_ratio:.2f} {status}")
    print(f"       → {gen_text[:80]}")

if all_pass:
    print("\n[INFO] LoRA 병합 후 검증 통과 → GPTQ 진행")
else:
    print("\n[WARN] ❌ 반복 루프 — LORA_STEPS 줄이거나 lr 낮춰서 재실행 권장")

[INFO] 생성 검증 (LoRA 병합 후)...
  [1] unique_ratio=1.00 ✅
       → 한국의 수도는 서울입니다.
  [2] unique_ratio=0.80 ✅
       → 정답은 **B. 지구는 둥글다**입니다. 

이 답변은 과학적 사실을 바탕으로 합니다. 인류의 관찰, 천문학적 관측, 다양한 과학적 실험 등
  [3] unique_ratio=0.65 ✅
       → Python에서 리스트를 정렬하는 함수는 `sorted()` 함수입니다. 이 함수는 리스트를 정렬하여 새로운 정렬된 리스트를 반환하며, 원본 리

[INFO] LoRA 병합 후 검증 통과 → GPTQ 진행


In [12]:
# =============================================
# GPTQ W4A16 양자화
# LoRA 병합된 가중치에 GPTQ 적용
# → 가중치가 이미 캘리브레이션 분포에 정렬됨 → 더 정확한 Hessian
# =============================================
print(f"[INFO] GPTQ 시작 (scheme={SCHEME}, samples={len(ds)})")

# ── GPTQ용 디바이스 설정 ─────────────────────
# llmcompressor oneshot()은 CUDA만 지원
# MPS 환경에서는 CPU로 이동 필요
if torch.cuda.is_available():
    model.to("cuda")
    print(f"[INFO] 양자화: CUDA GPU ({torch.cuda.get_device_name(0)})")
else:
    model.to("cpu")
    print("[INFO] 양자화: CPU 모드 (MPS는 llmcompressor 미지원)")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=len(ds),
)

print("[INFO] GPTQ 완료")

[INFO] GPTQ 시작 (scheme=W4A16, samples=160)
[INFO] 양자화: CPU 모드 (MPS는 llmcompressor 미지원)


Tokenizing:   0%|          | 0/160 [00:00<?, ? examples/s]

2026-02-25T16:32:26.707926+0900 | reset | INFO - Compression lifecycle reset
2026-02-25T16:32:26.708740+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-25T16:32:26.729121+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-25T16:32:26.729548+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-25T16:32:26.733918+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


(1/31): Calibrating: 100%|██████████| 160/160 [00:41<00:00,  3.90it/s]

2026-02-25T16:33:07.863682+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 160 samples


...

(30/31): Calibrating: 100%|██████████| 160/160 [00:42<00:00,  3.79it/s]

2026-02-25T17:03:11.495301+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 160 samples


2026-02-25T17:03:11.871517+0900 | compress | METRIC - time 0.38s
2026-02-25T17:03:11.871915+0900 | compress | METRIC - error 2703.76
2026-02-25T17:03:11.872652+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-25T17:03:11.872890+0900 | compress | METRIC - Compressed module size: 8.486912 MB
2026-02-25T17:03:11.874139+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 160 samples
2026-02-25T17:03:12.125307+0900 | compress | METRIC - time 0.25s
2026-02-25T17:03:12.125743+0900 | compress | METRIC - error 766.02
2026-02-25T17:03:12.126467+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-25T17:03:12.126666+0900 | compress | METRIC - Compressed module size: 2.121728 MB
2026-02-25T17:03:12.127381+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 160 samples
2026-02-25T17:03:12.376883+0900 | compress | METRIC - time 0.25s
2026-02-25T17:03

(31/31): Propagating: 100%|██████████| 160/160 [00:00<00:00, 1728.53it/s]


2026-02-25T17:03:35.010800+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-25T17:03:35.014691+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`
[INFO] GPTQ 완료


In [13]:
# =============================================
# 생성 검증 2 — GPTQ 후
# =============================================
print("[INFO] 생성 검증 (GPTQ 후)...")

model.eval()
if torch.cuda.is_available():
    gen_device = "cuda"
elif torch.backends.mps.is_available():
    gen_device = "mps"
else:
    gen_device = "cpu"
model.to(gen_device)

all_pass = True
for i, msgs in enumerate(test_prompts):
    prompt = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
    inputs = tokenizer(prompt, return_tensors="pt").to(gen_device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=40, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    gen_ids = out[0][inputs["input_ids"].shape[1]:]
    gen_text = tokenizer.decode(gen_ids, skip_special_tokens=True)
    unique_ratio = len(set(gen_ids.tolist())) / max(len(gen_ids), 1)
    status = "✅" if unique_ratio >= 0.3 else "❌ 반복루프"
    if unique_ratio < 0.3:
        all_pass = False
    print(f"  [{i+1}] unique_ratio={unique_ratio:.2f} {status}")
    print(f"       → {gen_text[:80]}")

if all_pass:
    print("\n[INFO] GPTQ 후 검증 통과 → 저장 진행")
else:
    print("\n[WARN] ❌ 반복 루프 — 양자화 오류 가능성. 저장은 진행하되 제출 주의")

[INFO] 생성 검증 (GPTQ 후)...
  [1] unique_ratio=1.00 ✅
       → 한국의 수도는 서울입니다.
  [2] unique_ratio=0.82 ✅
       → 정답은 **B. 지구는 둥글다**입니다. 

이 선택지들은 지구의 모양에 대한 다양한 관점을 제시하고 있습니다:
- A. 지구가 평평
  [3] unique_ratio=0.80 ✅
       → Python에서 리스트를 정렬하는 함수는 `sorted()` 함수입니다. 이 함수는 리스트의 요소들을 특정 순서(정렬)에 따라 반환하는 데 사용

[INFO] GPTQ 후 검증 통과 → 저장 진행


In [14]:
# =============================================
# 모델 저장
# =============================================
os.makedirs(OUT_DIR, exist_ok=True)

# tie_weights 복원 → lm_head 중복 저장 방지
if hasattr(model, "tie_weights"):
    model.tie_weights()
    print("[INFO] tie_weights() 완료")

model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

gc.collect()

# 파일 크기 검증
st = Path(OUT_DIR) / "model.safetensors"
size_mb = st.stat().st_size / 1e6 if st.exists() else 0
print(f"[INFO] 저장 완료: {OUT_DIR}")
print(f"  safetensors 크기: {size_mb:.0f} MB")
if size_mb > 1200:
    print("  ⚠ 1.2GB 초과 — lm_head 중복 저장 가능성 확인 필요")
else:
    print("  ✅ 크기 정상 (~930MB 예상)")

[INFO] tie_weights() 완료
2026-02-25T17:05:12.119012+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:02, 93.42it/s]


[INFO] 저장 완료: ./model
  safetensors 크기: 971 MB
  ✅ 크기 정상 (~930MB 예상)


In [15]:
# =============================================
# ZIP 생성 — 제출용
# =============================================
import zipfile

zip_name = "Try_017_lora_gptq"
out_path = Path(OUT_DIR).resolve()

print(f"[INFO] {zip_name}.zip 생성 중...")

# Colab: /content/drive/... 절대경로 사용 시 zip 내부 경로가 깨짐
# → root_dir=부모폴더, base_dir=폴더이름 방식으로 'model/' 루트 보장
if IS_COLAB:
    # Colab에서는 /content 아래에 임시 저장 후 다운로드
    zip_save_path = f"/content/{zip_name}"
else:
    zip_save_path = zip_name

shutil.make_archive(
    base_name=zip_save_path,
    format="zip",
    root_dir=str(out_path.parent),  # OUT_DIR 부모 (e.g. /content/drive/MyDrive 또는 .)
    base_dir=str(out_path.name),    # 'model'
)

final_zip = f"{zip_save_path}.zip"

# 내부 구조 검증
with zipfile.ZipFile(final_zip) as z:
    names = z.namelist()
    root_entries = {n.split("/")[0] for n in names if n.strip("/")}
    if "model" in root_entries:
        print(f"  ✅ 올바른 구조: model/ 루트 확인")
    else:
        print(f"  ❌ 구조 오류: {root_entries} — zip 경로 설정 확인 필요")

zip_mb = os.path.getsize(final_zip) / 1e6
print(f"[INFO] 생성 완료: {final_zip} ({zip_mb:.0f} MB)")

# Colab: 브라우저 다운로드 트리거
if IS_COLAB:
    from google.colab import files
    print(f"[INFO] Colab 다운로드 시작: {final_zip}")
    files.download(final_zip)

[INFO] Try_017_lora_gptq.zip 생성 중...
  ✅ 올바른 구조: model/ 루트 확인
[INFO] 생성 완료: Try_017_lora_gptq.zip (814 MB)
